# Aggregations and CTEs

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/sql/sql-02-aggregations-and-ctes.ipynb)

The basics chapter introduced `COUNT`, `SUM` and `GROUP BY`. This chapter goes further. It covers the aggregates
you will use most, grouping at several levels, subqueries that feed a query, and **common table expressions**
(CTEs), which name each step of a long query so it reads from top to bottom.

It ends with practice questions and answers.

## Setup

Install PySpark, start a Spark session, and create the example tables.

In [ ]:
!pip install -q pyspark

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

In [ ]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW customers AS
SELECT * FROM VALUES
  (1, 'Acme Corp',    'North', DATE '2022-01-05'),
  (2, 'Beta LLC',     'North', DATE '2022-02-10'),
  (3, 'Cyan Inc',     'South', DATE '2022-03-15'),
  (4, 'Delta Co',     'South', DATE '2022-04-20'),
  (5, 'Echo Ltd',     'West',  DATE '2022-05-25'),
  (6, 'Foxtrot GmbH', 'West',  DATE '2022-06-30'),
  (7, 'Gamma SA',     'East',  DATE '2023-01-10'),
  (8, 'Hotel BV',     'East',  DATE '2023-02-15')
AS customers(customer_id, name, region, signup_date)
""")

In [ ]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW orders AS
SELECT * FROM VALUES
  (1,  1, 'completed', 100, DATE '2023-01-05'),
  (2,  1, 'completed', 200, DATE '2023-02-10'),
  (3,  1, 'completed', 150, DATE '2023-03-15'),
  (4,  2, 'completed', 300, DATE '2023-01-20'),
  (5,  2, 'cancelled', 250, DATE '2023-02-05'),
  (6,  2, 'completed', 400, DATE '2023-03-20'),
  (7,  3, 'completed', 120, DATE '2023-01-25'),
  (8,  3, 'completed', 180, DATE '2023-02-28'),
  (9,  4, 'pending',   220, DATE '2023-02-14'),
  (10, 4, 'completed', 320, DATE '2023-03-30'),
  (11, 5, 'completed', 500, DATE '2023-01-02'),
  (12, 5, 'completed', 450, DATE '2023-03-05'),
  (13, 6, 'completed', 275, DATE '2023-02-18'),
  (14, 6, 'cancelled', 325, DATE '2023-03-25')
AS orders(order_id, customer_id, status, amount, order_date)
""")

These are the same two tables as in the basics chapter. `customers` has 8 rows across four regions; customers 7
and 8 (region East) have no orders. `orders` has 14 rows: 11 completed, 2 cancelled, 1 pending, totalling 3,790.
Joined to their customers, the orders fall into three regions:

| region | customers | orders | total of all orders |
| --- | --- | --- | --- |
| North | 1, 2 | 6 | 1,400 |
| South | 3, 4 | 4 | 840 |
| West | 5, 6 | 4 | 1,550 |

In [ ]:
q = """
SELECT c.region, COUNT(*) AS orders, SUM(o.amount) AS total
FROM orders o
JOIN customers c ON c.customer_id = o.customer_id
GROUP BY c.region
ORDER BY c.region
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [("North", 6, 1400), ("South", 4, 840), ("West", 4, 1550)]

## 1. More aggregate functions

`COUNT(DISTINCT column)` counts the different non-NULL values in a column. Compare it with `COUNT(column)`, which
counts every non-NULL value, repeats included: `orders` has 14 values of `customer_id` but only 6 different ones.

An aggregate can take an expression, not only a column name, so `SUM(amount * 0.1)` sums ten per cent of each
order. `MIN` and `MAX` also work on dates and text, which makes them the usual tools for "first seen" and
"last seen".

### Where you'll use it

- **Unique users:** `COUNT(DISTINCT user_id)` for monthly active users, the most common distinct count in product analytics.
- **Percentiles:** median order value or p95 latency for service-level reporting.
- **Conditional counts:** completed and cancelled orders side by side in one row of a status report.

Worked by hand: the earliest order is order 11 (2023-01-02) and the latest is order 10 (2023-03-30). All 14
amounts sum to 3,790, so ten per cent is 379.0. `0.1` is a `DECIMAL` literal, so the result is a `Decimal`,
and the check converts it with `float`.

In [ ]:
q = """
SELECT
  COUNT(customer_id)          AS customer_values,
  COUNT(DISTINCT customer_id) AS buyers,
  COUNT(DISTINCT status)      AS statuses,
  MIN(order_date)             AS first_order,
  MAX(order_date)             AS last_order,
  ROUND(SUM(amount * 0.1), 2) AS tenth_of_all_amounts
FROM orders
"""
spark.sql(q).show()

from datetime import date

row = spark.sql(q).first()
assert tuple(row[:5]) == (14, 6, 3, date(2023, 1, 2), date(2023, 3, 30))
assert float(row.tenth_of_all_amounts) == 379.0

### Medians and percentiles

The **median** is the middle value after sorting; half the values are below it. A **percentile** generalises
this: the 95th percentile (p95) is the value below which 95 per cent of the values fall.

The 14 amounts sorted are 100, 120, 150, 180, 200, 220, **250, 275**, 300, 320, 325, 400, 450, 500. With an even
count there are two middle values, and the exact median is their average: (250 + 275) / 2 = 262.5.

- `percentile(amount, 0.5)` and `median(amount)` compute the exact value, 262.5.
- `percentile_approx(amount, 0.5)` returns a value from the data that is close to it. It uses far less memory on
  large tables, which is why dashboards over billions of rows use it.

In [ ]:
q = """
SELECT percentile(amount, 0.5)        AS exact_median,
       median(amount)                 AS median,
       percentile_approx(amount, 0.5) AS approximate_median
FROM orders
"""
spark.sql(q).show()
exact, median, approximate = spark.sql(q).first()
assert exact == median == 262.5
assert approximate in {250, 275}

### Counting only some rows: FILTER

`COUNT(*) FILTER (WHERE condition)` applies an aggregate to only the rows that pass the condition. It does the
same job as `SUM(CASE WHEN condition THEN 1 ELSE 0 END)` from the basics chapter, and reads more directly. Each
aggregate in the `SELECT` can have its own filter, so one pass over the table gives several counts.

In [ ]:
q = """
SELECT COUNT(*)                                          AS all_orders,
       COUNT(*) FILTER (WHERE status = 'completed')      AS completed,
       COUNT(*) FILTER (WHERE status = 'cancelled')      AS cancelled,
       SUM(amount) FILTER (WHERE status = 'completed')   AS completed_amount
FROM orders
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [(14, 11, 2, 2995)]

## 2. Grouping at several levels

Listing more than one column after `GROUP BY` creates one group per combination, which is useful for drill-downs.

Each set of columns you group by is a **grouping set**. A plain `GROUP BY region, status` has one grouping set,
`(region, status)`. `ROLLUP` adds coarser grouping sets in one query: `GROUP BY ROLLUP(a, b)` is short for
`GROUPING SETS ((a, b), (a), ())`. That gives the detail rows for each `(a, b)` pair, one subtotal row per `a`,
and a grand total over the empty set `()`. In a subtotal row, the column that was rolled up is `NULL`.

### Where you'll use it

- **Financial reports:** totals by region, then by region and product, with a grand total on the last line.
- **Dashboards with drill-down:** one query returns the subtotals and the detail together.

The detail level has six `(region, status)` pairs. North, for example, has completed orders 1, 2, 3, 4 and 6
(100 + 200 + 150 + 300 + 400 = 1,150) and cancelled order 5 (250).

In [ ]:
q = """
SELECT c.region, o.status, COUNT(*) AS orders, SUM(o.amount) AS total
FROM orders o
JOIN customers c ON c.customer_id = o.customer_id
GROUP BY c.region, o.status
ORDER BY c.region, o.status
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [
    ("North", "cancelled", 1, 250), ("North", "completed", 5, 1150),
    ("South", "completed", 3, 620), ("South", "pending", 1, 220),
    ("West", "cancelled", 1, 325), ("West", "completed", 3, 1225),
]

`ROLLUP(region, status)` returns those 6 detail rows, plus 3 region subtotals (North 1,400, South 840,
West 1,550), plus 1 grand total (3,790): 10 rows. Spark sorts `NULL` first in ascending order, so the grand total
comes first and each region's subtotal comes before its detail rows.

In [ ]:
q = """
SELECT c.region, o.status, SUM(o.amount) AS total
FROM orders o
JOIN customers c ON c.customer_id = o.customer_id
GROUP BY ROLLUP(c.region, o.status)
ORDER BY c.region, o.status
"""
spark.sql(q).show()
rollup_rows = spark.sql(q).collect()
assert rollup_rows == [
    (None, None, 3790),
    ("North", None, 1400), ("North", "cancelled", 250), ("North", "completed", 1150),
    ("South", None, 840), ("South", "completed", 620), ("South", "pending", 220),
    ("West", None, 1550), ("West", "cancelled", 325), ("West", "completed", 1225),
]

grouping_sets = spark.sql("""
SELECT c.region, o.status, SUM(o.amount) AS total
FROM orders o
JOIN customers c ON c.customer_id = o.customer_id
GROUP BY GROUPING SETS ((c.region, o.status), (c.region), ())
ORDER BY c.region, o.status
""").collect()
assert grouping_sets == rollup_rows

### Telling subtotals apart: GROUPING()

In a `ROLLUP` result, a NULL can mean a real NULL in the data, or a subtotal row. `GROUPING(column)` returns 1
when the row is a subtotal over that column and 0 when the column holds a real grouped value. Use it to label the
rows explicitly, and to sort each subtotal after its detail rows.

In [ ]:
q = """
SELECT region, status, total
FROM (
  SELECT CASE WHEN GROUPING(c.region) = 1 THEN 'all regions' ELSE c.region END AS region,
         CASE WHEN GROUPING(o.status) = 1 THEN 'all statuses' ELSE o.status END AS status,
         SUM(o.amount) AS total,
         GROUPING(c.region) AS region_total,
         GROUPING(o.status) AS status_total
  FROM orders o
  JOIN customers c ON c.customer_id = o.customer_id
  GROUP BY ROLLUP(c.region, o.status)
) t
ORDER BY region_total, region, status_total, status
"""
spark.sql(q).show(truncate=False)
assert spark.sql(q).collect() == [
    ("North", "cancelled", 250), ("North", "completed", 1150), ("North", "all statuses", 1400),
    ("South", "completed", 620), ("South", "pending", 220), ("South", "all statuses", 840),
    ("West", "cancelled", 325), ("West", "completed", 1225), ("West", "all statuses", 1550),
    ("all regions", "all statuses", 3790),
]

### CUBE: every combination

`ROLLUP` only removes columns from the right, so it never produces a total per `status` across all regions.
`CUBE(a, b)` produces every grouping set: `(a, b)`, `(a)`, `(b)` and `()`. Here that adds three status totals
to the ten `ROLLUP` rows: cancelled 575, completed 2,995 and pending 220, for 13 rows.

In [ ]:
q = """
SELECT c.region, o.status, SUM(o.amount) AS total
FROM orders o
JOIN customers c ON c.customer_id = o.customer_id
GROUP BY CUBE(c.region, o.status)
ORDER BY c.region, o.status
"""
cube_rows = spark.sql(q).collect()
spark.sql(q).show()
assert len(cube_rows) == 13
assert cube_rows[:4] == [(None, None, 3790), (None, "cancelled", 575), (None, "completed", 2995), (None, "pending", 220)]

## 3. Subqueries

A **subquery** is a query inside another query, written in parentheses. The query that contains it is the
**outer** query. Where the subquery appears decides what it must return.

- A **scalar subquery** returns one value (one row, one column), and can stand wherever a value can.
- An **IN** subquery returns one column of values to test against.
- An **EXISTS** subquery is true when it returns at least one row, false otherwise. Its columns do not matter.
- A **correlated** subquery refers to a column of the outer query's current row, so logically it runs once per
  outer row. An uncorrelated subquery does not, and runs once in total.

### Where you'll use it

- **Comparing a row to its group:** orders larger than the customer's own average.
- **Existence checks:** customers with at least one refund, using `EXISTS`.

The first query below uses a correlated scalar subquery in `SELECT`: for each customer, count that customer's
orders. For Gamma SA the subquery finds no rows, and `COUNT` of no rows is 0.

In [ ]:
q = """
SELECT c.name,
       (SELECT COUNT(*) FROM orders o WHERE o.customer_id = c.customer_id) AS orders
FROM customers c
ORDER BY c.customer_id
"""
spark.sql(q).show()
assert [row.orders for row in spark.sql(q).collect()] == [3, 3, 2, 2, 2, 2, 0, 0]

An `IN` subquery: the inner query returns the customer IDs of cancelled orders, 2 and 6. It does not refer to
the outer row, so it is uncorrelated.

In [ ]:
q = """
SELECT name FROM customers
WHERE customer_id IN (SELECT customer_id FROM orders WHERE status = 'cancelled')
ORDER BY name
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [("Beta LLC",), ("Foxtrot GmbH",)]

### A correlated subquery, worked by hand

The next query keeps orders larger than **their own customer's** average. For each outer order `o`, the subquery
averages the orders `x` with the same `customer_id`:

| customer | amounts | average | orders above it |
| --- | --- | --- | --- |
| 1 | 100, 200, 150 | 150 | 2 (200) |
| 2 | 300, 250, 400 | 316.67 | 6 (400) |
| 3 | 120, 180 | 150 | 8 (180) |
| 4 | 220, 320 | 270 | 10 (320) |
| 5 | 500, 450 | 475 | 11 (500) |
| 6 | 275, 325 | 300 | 14 (325) |

The two table aliases `o` and `x` are needed because both refer to `orders`; `x.customer_id = o.customer_id` is
what ties the inner query to the outer row.

In [ ]:
q = """
SELECT order_id, customer_id, amount
FROM orders o
WHERE amount > (SELECT AVG(x.amount) FROM orders x WHERE x.customer_id = o.customer_id)
ORDER BY order_id
"""
spark.sql(q).show()
assert [row.order_id for row in spark.sql(q).collect()] == [2, 6, 8, 10, 11, 14]

### EXISTS and NOT EXISTS

`EXISTS` asks "is there at least one matching row?". The engine can stop at the first match, and a customer
appears once however many orders match. `NOT EXISTS` asks the opposite and is the usual way to find rows with
no match: here, customers 7 and 8.

In [ ]:
q = """
SELECT c.name
FROM customers c
WHERE NOT EXISTS (SELECT 1 FROM orders o WHERE o.customer_id = c.customer_id)
ORDER BY c.name
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [("Gamma SA",), ("Hotel BV",)]

### Why NOT IN is risky

`x NOT IN (1, NULL)` means `x <> 1 AND x <> NULL`. The second comparison is unknown for every `x`, so the whole
condition is never true, and the query returns no rows. If an `IN` subquery can return a `NULL`, `NOT IN`
silently returns nothing. `NOT EXISTS` does not have this problem.

In [ ]:
q = """
SELECT name FROM customers
WHERE customer_id NOT IN (SELECT 1 UNION ALL SELECT NULL)
"""
assert spark.sql(q).collect() == []

q = """
SELECT name FROM customers
WHERE customer_id NOT IN (SELECT 1)
"""
assert spark.sql(q).count() == 7

A subquery in the `FROM` clause is also common. It is called a **derived table**: the inner query produces a
table and the outer query reads it like any other. Here it computes each customer's total over all statuses
(1: 450, 2: 950, 3: 300, 4: 540, 5: 950, 6: 600), and the outer query keeps totals of at least 500. The outer
`WHERE` can filter on `total` because, for the outer query, `total` is an ordinary column.

In [ ]:
q = """
SELECT customer_id, total
FROM (SELECT customer_id, SUM(amount) AS total FROM orders GROUP BY customer_id) t
WHERE total >= 500
ORDER BY customer_id
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [(2, 950), (4, 540), (5, 950), (6, 600)]

## 4. Common table expressions (CTEs)

A CTE is a named query placed before the main query with `WITH`:

```sql
WITH step1 AS (...),
     step2 AS (...)
SELECT ... FROM step2;
```

Each step can use the steps before it. The main query reads the last step. Long queries become a sequence of named
steps, and each step can be checked on its own. A CTE is visible only inside the query that defines it.

The query below answers "which regions had more than 1,000 in completed orders?" in three named steps:

### Where you'll use it

- **Readable transformations:** a dbt model is a chain of CTEs, one per logical step, that a reviewer can follow.
- **Reusing an intermediate result:** a CTE of completed orders used in two later steps.

In [ ]:
q = """
WITH completed AS (
  SELECT customer_id, amount
  FROM orders
  WHERE status = 'completed'
),
by_customer AS (
  SELECT customer_id, SUM(amount) AS total
  FROM completed
  GROUP BY customer_id
),
by_region AS (
  SELECT c.region, SUM(b.total) AS total
  FROM by_customer b
  JOIN customers c ON c.customer_id = b.customer_id
  GROUP BY c.region
)
SELECT region, total
FROM by_region
WHERE total > 1000
ORDER BY region
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [("North", 1150), ("West", 1225)]

### Checking one step at a time

To debug a CTE chain, point the main query at an earlier step. The query below keeps the first two steps and
selects from `by_customer`, so you can confirm the per-customer totals (1: 450, 2: 700, 3: 300, 4: 320, 5: 950,
6: 275) before trusting the region step. South's total of 620 is then 300 + 320, and it fails the `> 1000` test.

In [ ]:
q = """
WITH completed AS (
  SELECT customer_id, amount FROM orders WHERE status = 'completed'
),
by_customer AS (
  SELECT customer_id, SUM(amount) AS total FROM completed GROUP BY customer_id
)
SELECT customer_id, total FROM by_customer ORDER BY customer_id
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [(1, 450), (2, 700), (3, 300), (4, 320), (5, 950), (6, 275)]

### A CTE exists only inside its query

A CTE is not a table and is not saved anywhere. Once the statement ends, the name is gone, so a later query that
reads `completed` is rejected. To share a step across queries, create a view instead. Most engines, Spark
included, do not store a CTE's result: if the main query references a CTE twice, its query may be computed twice.

In [ ]:
from pyspark.errors import AnalysisException

rejected = False
try:
    spark.sql("SELECT * FROM completed")
except AnalysisException as error:
    rejected = True
    print(str(error)[:80])
assert rejected

### Recursive CTEs

A CTE can refer to itself. That is how SQL walks a hierarchy, such as an employee's chain of managers. A recursive
CTE has two parts joined by `UNION ALL`: an **anchor** query that returns the starting rows, then a **recursive**
step that joins the rows found so far to the table to find the next level down. The engine repeats the step until
it adds no new rows. The `staff` table below has a `manager_id` for each person.

Spark 4.0, the version Colab installs today, fails to plan a recursive CTE with an internal error. The query is
therefore shown as a reference and is not run here. The example below walks the same hierarchy with a self-join,
which works on every version and covers a fixed number of levels.

### Where you'll use it

- **Org charts and bills of materials:** walking a manager chain or a parts hierarchy.
- **Graph traversal:** finding all accounts linked to a flagged account through several hops.

In [ ]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW staff AS
SELECT * FROM VALUES
  (1, 'Alice', NULL),
  (2, 'Bob',   1),
  (3, 'Carol', 1),
  (4, 'Dave',  2),
  (5, 'Eve',   2)
AS staff(emp_id, name, manager_id)
""")

Reference only, not executed. Spark 4.1 and later run this form of recursive CTE:

```sql
WITH RECURSIVE reports(emp_id, name, level) AS (
  SELECT emp_id, name, 0 FROM staff WHERE name = 'Alice'
  UNION ALL
  SELECT s.emp_id, s.name, r.level + 1
  FROM staff s
  JOIN reports r ON s.manager_id = r.emp_id
)
SELECT name, level FROM reports WHERE level > 0 ORDER BY emp_id
```

Traced by hand: the anchor returns Alice at level 0. The first step finds everyone whose manager is Alice: Bob
and Carol at level 1. The second step finds everyone whose manager is Bob or Carol: Dave and Eve at level 2. The
third step finds nobody, so the recursion stops.

### The self-join version

A **self-join** joins a table to itself under two aliases, so one copy plays the employee and the other the
manager. Each extra level needs one more join, which is why this version only reaches a fixed depth. The first
branch finds direct reports of Alice; the second goes through one middle manager.

In [ ]:
q = """
SELECT s.emp_id, s.name, 1 AS level
FROM staff s
JOIN staff boss ON s.manager_id = boss.emp_id
WHERE boss.name = 'Alice'
UNION ALL
SELECT s.emp_id, s.name, 2
FROM staff s
JOIN staff mid ON s.manager_id = mid.emp_id
JOIN staff boss ON mid.manager_id = boss.emp_id
WHERE boss.name = 'Alice'
ORDER BY emp_id
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [(2, "Bob", 1), (3, "Carol", 1), (4, "Dave", 2), (5, "Eve", 2)]

## 5. Choosing between a join, a subquery and a CTE

| Use | When |
| --- | --- |
| A **join** | You need columns from both tables in the result. |
| A **subquery** or **EXISTS** | You only need to test for a match, or you need one value per row. |
| A **CTE** | The logic has several named steps, or you use the same intermediate result more than once. |

A CTE does not change the answer, only how readable the query is. When a subquery and a CTE give the same result,
prefer the one a reader can follow more easily.

### Where you'll use it

- **Code review:** the team's SQL style guide usually asks for CTEs for multi-step logic, because a reviewer can read them top to bottom.

### A join repeats rows; EXISTS does not

"Which customers have a completed order?" is a test for a match. A join answers it with one row per matching
**order**: Acme Corp has three completed orders and appears three times, 11 rows in all. `EXISTS` returns each
customer once: 6 rows. Use a join only when you need the order columns too.

In [ ]:
joined = spark.sql("""
SELECT c.name
FROM customers c
JOIN orders o ON o.customer_id = c.customer_id AND o.status = 'completed'
""")
tested = spark.sql("""
SELECT c.name
FROM customers c
WHERE EXISTS (SELECT 1 FROM orders o WHERE o.customer_id = c.customer_id AND o.status = 'completed')
""")
print(joined.count(), tested.count())
assert joined.count() == 11
assert tested.count() == 6
assert joined.distinct().count() == tested.count()

## Interview notes

- **Group before you join when you can.** Aggregating a large table first and joining the small result is usually
  cheaper than joining everything and then aggregating.
- **COUNT(DISTINCT) is expensive.** It must track every distinct value. For very large data, engines offer
  approximate versions, such as `approx_count_distinct`, and `percentile_approx` for percentiles.
- **ROLLUP vs CUBE.** `ROLLUP(a, b)` gives `(a, b)`, `(a)`, `()`; `CUBE(a, b)` adds `(b)`. Use `GROUPING()` to
  tell a subtotal's NULL from a real one.
- **NOT IN vs NOT EXISTS.** `NOT IN` returns nothing when the list contains a NULL. Prefer `NOT EXISTS`.
- **A correlated subquery runs per row** in the logical model. Engines often rewrite it into a join, but a
  correlated subquery you cannot rewrite can be slow; prefer a join or a CTE that computes the value once per group.
- **CTEs are for readability.** A CTE is usually not stored; it is expanded into the main query.

## Summary

- Use `COUNT(DISTINCT ...)`, `percentile` and `median`, expressions inside aggregates, and `FILTER` for
  conditional counts.
- `ROLLUP`, `CUBE` and `GROUPING SETS` compute several grouping levels in one query; `GROUPING()` labels them.
- A correlated subquery refers to the outer row. `EXISTS` tests for a match; `NOT EXISTS` is the safe
  opposite of `NOT IN`.
- A CTE names each step and lives only inside its query. A recursive CTE walks a hierarchy on versions that
  support it; a self-join covers a fixed depth.

## Practice questions

1. How many customers have a completed order, and when was the latest completed order placed? Show both in one row.
2. Which orders are larger than the average amount of orders with the **same status**? Show `order_id`, `status`
   and `amount`, sorted by `order_id`.
3. Which customers have at least one cancelled order? Use `EXISTS`, and show their names sorted by name.
4. Using CTEs, find the customers whose completed total is above the average completed total per customer. Show
   `customer_id` and the total, sorted by `customer_id`.
5. Using a self-join, list every employee with their manager's name. Alice has no manager and should still
   appear, with `NULL`. Sort by `emp_id`.
6. Show the completed total for each region plus a grand total labelled `all regions`, with the grand total last.

## Answers

### Question 1

Filter to completed orders, then use `COUNT(DISTINCT ...)` for customers and `MAX` for the latest date. All six
customers with orders have a completed one; the latest is order 10 on 2023-03-30.

In [ ]:
q = """
SELECT COUNT(DISTINCT customer_id) AS customers, MAX(order_date) AS latest_order
FROM orders
WHERE status = 'completed'
"""
spark.sql(q).show()

from datetime import date

assert spark.sql(q).collect() == [(6, date(2023, 3, 30))]

### Question 2

A correlated subquery, as in section 3, but correlated on `status` instead of `customer_id`. The averages are
completed 2995 / 11 = 272.27, cancelled 287.5 and pending 220. Completed orders above 272.27 are 4, 6, 10, 11,
12 and 13 (275); cancelled order 14 (325) is above 287.5; the pending order equals its own average, so it is not
larger.

In [ ]:
q = """
SELECT order_id, status, amount
FROM orders o
WHERE amount > (SELECT AVG(x.amount) FROM orders x WHERE x.status = o.status)
ORDER BY order_id
"""
spark.sql(q).show()
assert [row.order_id for row in spark.sql(q).collect()] == [4, 6, 10, 11, 12, 13, 14]

### Question 3

`EXISTS` with a subquery correlated on `customer_id` and filtered to cancelled orders.

In [ ]:
q = """
SELECT c.name
FROM customers c
WHERE EXISTS (SELECT 1 FROM orders o WHERE o.customer_id = c.customer_id AND o.status = 'cancelled')
ORDER BY c.name
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [("Beta LLC",), ("Foxtrot GmbH",)]

### Question 4

One CTE computes each customer's completed total; a second reduces it to a single average; the main query
compares the two. The totals are 450, 700, 300, 320, 950 and 275, whose average is 2995 / 6 = 499.17. Only
customers 2 and 5 are above it. The first CTE is used twice, which is where a CTE helps most.

In [ ]:
q = """
WITH by_customer AS (
  SELECT customer_id, SUM(amount) AS total
  FROM orders
  WHERE status = 'completed'
  GROUP BY customer_id
),
benchmark AS (
  SELECT AVG(total) AS average_total FROM by_customer
)
SELECT b.customer_id, b.total
FROM by_customer b
CROSS JOIN benchmark
WHERE b.total > benchmark.average_total
ORDER BY b.customer_id
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [(2, 700), (5, 950)]

### Question 5

Join `staff` to itself: `s` is the employee, `m` the manager. A `LEFT JOIN` keeps Alice, whose `manager_id` is
`NULL` and so matches nobody.

In [ ]:
q = """
SELECT s.emp_id, s.name, m.name AS manager
FROM staff s
LEFT JOIN staff m ON s.manager_id = m.emp_id
ORDER BY s.emp_id
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [
    (1, "Alice", None), (2, "Bob", "Alice"), (3, "Carol", "Alice"), (4, "Dave", "Bob"), (5, "Eve", "Bob"),
]

### Question 6

`ROLLUP` on one column gives the per-region rows and a grand total. `GROUPING()` labels the grand total and,
used as the first sort key, puts it last. Completed totals are North 1,150, South 620 and West 1,225, and
together 2,995.

In [ ]:
q = """
SELECT region, completed_total
FROM (
  SELECT CASE WHEN GROUPING(c.region) = 1 THEN 'all regions' ELSE c.region END AS region,
         SUM(o.amount) AS completed_total,
         GROUPING(c.region) AS is_total
  FROM orders o
  JOIN customers c ON c.customer_id = o.customer_id
  WHERE o.status = 'completed'
  GROUP BY ROLLUP(c.region)
) t
ORDER BY is_total, region
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [("North", 1150), ("South", 620), ("West", 1225), ("all regions", 2995)]